# pSMAD_2024-08-21 — 01_manifest_qc

**Feeds:** ED Fig 2j, 2k

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Manifest QC

Confirm the raw dataset-2 channel files share the expected tile lattice, pixel size, and channel identities before restitching.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import chip_mosaic_helpers as cmh

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent.resolve()

RAW_DIR = ROOT / '2024-08-21_pSMAD'
MANIFEST_DIR = ROOT / 'results/manifests'
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = MANIFEST_DIR / 'dataset2_czi_manifest.csv'
OUT_JSON = MANIFEST_DIR / 'dataset2_czi_manifest.json'

FILE_MAP = {
    'brightfield': RAW_DIR / 'well3-bright.czi',
    'dapi': RAW_DIR / 'well3-DAPI.czi',
    'bead': RAW_DIR / 'well3-555.czi',
    'psmad': RAW_DIR / 'well3-pSMAD.czi',
}

print('Project root:', ROOT)
print('Raw dir:', RAW_DIR)


## Build Channel Manifest

In [ ]:
rows = []
infos = {}
for logical_channel, path in FILE_MAP.items():
    if not path.exists():
        raise RuntimeError(f'Missing raw file for {logical_channel}: {path}')
    info = cmh.inspect_single_channel_czi(path)
    infos[logical_channel] = info
    rows.append({
        'logical_channel': logical_channel,
        'file_name': path.name,
        'axes': info.axes,
        'shape': str(info.shape),
        'scene_count': info.scene_count,
        'z_count': len(info.z_values),
        'tile_rows': len(info.y_starts),
        'tile_cols': len(info.x_starts),
        'tile_shape_yx': str(info.tile_shape_yx),
        'nominal_dy_px': info.nominal_dy_px,
        'nominal_dx_px': info.nominal_dx_px,
        'scale_x_um': info.scale_x_um,
        'scale_y_um': info.scale_y_um,
        'channel_name_from_xml': cmh.channel_name_from_xml(cmh.read_embedded_xml(path)),
    })
manifest_df = pd.DataFrame(rows)
display(manifest_df)


## Cross-Channel Consistency Checks

In [ ]:
ref = infos['brightfield']
mismatches = []
for logical_channel, info in infos.items():
    if info.y_starts != ref.y_starts:
        mismatches.append((logical_channel, 'y_starts'))
    if info.x_starts != ref.x_starts:
        mismatches.append((logical_channel, 'x_starts'))
    if info.scene_count != ref.scene_count:
        mismatches.append((logical_channel, 'scene_count'))
    if info.tile_shape_yx != ref.tile_shape_yx:
        mismatches.append((logical_channel, 'tile_shape_yx'))
if mismatches:
    raise RuntimeError(f'Channel tile lattice mismatch detected: {mismatches}')
print('All channels share the same tile lattice and scene count.')
print('Tile rows x cols:', len(ref.y_starts), 'x', len(ref.x_starts))
print('Z counts:', {k: len(v.z_values) for k, v in infos.items()})


## Save Stage Outputs

In [ ]:
manifest_df.to_csv(OUT_CSV, index=False)
OUT_JSON.write_text(manifest_df.to_json(orient='records', indent=2), encoding='utf-8')
print('Wrote CSV:', OUT_CSV)
print('Wrote JSON:', OUT_JSON)
